# 02 — Time-aware preprocessing

Deduplicate, resample hourly, forward-fill sensors within a limit and keep missing targets so no fake ground truth is created.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]

from cooling_load.config import load_config
from cooling_load.io import read_frame
from cooling_load.pipeline import run_preprocessing

config = load_config(ROOT / "configs/base.yaml")
raw = read_frame(config.path("raw_data"))
clean = run_preprocessing(config, raw)

In [ ]:
quality = {
    "raw_rows": len(raw),
    "clean_rows": len(clean),
    "duplicate_keys_after": int(clean.duplicated(["building_id", "timestamp"]).sum()),
    "target_missing": int(clean.cooling_load_kwh.isna().sum()),
}
quality

In [ ]:
display(clean.head())
display(clean.isna().mean().sort_values(ascending=False).to_frame("missing_fraction"))
print("Saved:", config.path("interim_data"))